# Case study: trimming redundant features before a model

A churn table contains nine columns. The modeling team
wants to know which columns carry their own information — and which are
derived, duplicated, or concentrated. The profiler measures which
columns are derivable from others, how their values are distributed, and
which columns predict the target. The modeling team can use these results
to review candidate features.

In [1]:
from html import escape

from IPython.display import HTML, display

import random

import polars as pl

import onceover

rng = random.Random(77)
n = 5_000

tenure = [rng.randrange(0, 60) for _ in range(n)]
tenure_years = [round(t / 12, 2) for t in tenure]          # rounded copy
x1 = [rng.uniform(-1, 1) for _ in range(n)]
x2 = [rng.uniform(-1, 1) for _ in range(n)]
score = [round(a * b + rng.uniform(-0.05, 0.05), 4) for a, b in zip(x1, x2)]  # interaction
channel = ["web" if rng.random() < 0.98 else "app" for _ in range(n)]          # concentrated
segment = [rng.choice(["retail", "b2b"]) for _ in range(n)]
plan = [["starter", "scale"][s == "b2b"] if rng.random() < 0.97
        else ["starter", "scale"][1 - (s == "b2b")]
        for s in segment]                                    # 3% violations
# new customers churn: tenure under a year, with 5% label noise
churn = [(rng.random() < 0.95) == (t < 12) for t in tenure]

df = pl.DataFrame({
    "tenure": tenure,
    "tenure_years": tenure_years,
    "x1": x1,
    "x2": x2,
    "score": score,
    "channel": channel,
    "segment": segment,
    "plan": plan,
    "churn": churn,
})

report = onceover.profile(df)
display(HTML(f"<p>{escape(report.overview['headline'])}</p>"))

The headline reports functional dependencies, predictable columns, and
rules. Its `key candidate: x1` needs context: `x1` is a random float,
unique and non-null in all 5,000 rows. It meets the observed key criteria
but does not represent an entity identifier.

For feature trimming, inspect dependencies for derived columns,
concentration for columns with little variation, and predictive
statements for candidate inputs.

In [2]:
print("— dependencies")
for f in report.findings:
    if f["category"] == "dependencies":
        print(f"  [{f['type']}] {f['claim']}")
print()
print("— concentration and roles")
for f in report.findings:
    if f["category"] in {"concentration", "column_roles"}:
        print(f"  {f['claim']}")
print()
print("— rules")
for f in report.findings:
    if f["category"] == "conditional_rules":
        print(f"  {f['claim']}")

— dependencies
  [predictive] score follows 8 local linear regimes (first split x1 ≤ -0.694521; holdout R² 0.887 versus -0.001 for one linear model)
  [predictive] tenure is predictable from tenure_years (100% of gain) (holdout R² = 1.00 using only these drivers; 1.00 using all inputs); tenure's median rises with tenure_years: 59 where tenure_years > 4.83 against 1 where tenure_years <= 0.08
  [predictive] plan is predictable from segment (100% of gain) (holdout accuracy 0.97 vs 0.52 majority baseline); the share of rows with plan = 'starter' is highest where segment = 'retail' (97%) and lowest where segment = 'b2b' (4%)
  [predictive] segment is predictable from plan (100% of gain) (holdout accuracy 0.97 vs 0.51 majority baseline); the share of rows with segment = 'retail' is highest where plan = 'starter' (96%) and lowest where plan = 'scale' (3%)
  [predictive] score is predictable from x1 (58% of gain), x2 (42% of gain) (holdout R² = 0.98 using only these drivers; 0.98 using all in

## The trimming checklist

**Redundant measurements.** `tenure` and `tenure_years` are exact
functions of each other: 0 violations in both directions. Because
`tenure_years` is `tenure / 12` rounded to two decimals, every observed
integer month count maps to one year value and back, so the profiler's
partition test finds a bijection rather than fitting a slope. These two columns contain the same observed information. Keeping both
can divide feature importance between equivalent inputs. Decision: drop `tenure_years`, keep `tenure`.

**Check what a unique float represents.** `x1` and `x2` both surface as
identifier-role columns, and `x1` wins the headline's key ranking. In a
feature table that is a warning: the column is unique because it is
continuous, not because it identifies an entity. Keep it as a feature;
do not join on it.

**Interactions are not redundancy.** `score` is predictable from `x2` and `x1` at holdout R² 0.98 — the
statement names those two drivers because the model's gain sits almost
entirely on them (gain shares 0.63 and 0.37, respectively; every other
input is below 0.001), and a refit on the two drivers alone reaches the
same 0.98 as the full input set. The piecewise model makes the same point from another angle:
`score` follows 8 local linear regimes with holdout R² 0.910, against
0.002 for one global linear model. The low global linear R² misses the interaction used to construct
`score`. Keep `x1` and `x2` as candidate inputs and compare models with
an explicit product feature.

**Review concentrated columns.** `channel` is 98.1% one
value — effective cardinality 1.10. Most rows therefore have the same encoded value. Exclude it in this
fixture, but check the minority rows and predictive performance before
dropping a similarly concentrated column in a real dataset.

**Inspect approximate mappings.** The rules from `segment` to `plan`
hold in 96–98% of matching rows depending on the segment (b2b → scale 96.03%,
retail → starter 98.41%); the profiler also lists `segment, score → plan`
at 99.22% with 39 violations. The additional condition needs scrutiny: `score` is a near-unique float, so putting it in the antecedent
can leave few repeated input combinations. Check support and violations before treating these mappings as
constraints or using them to remove features.

**The target is predictable.** The model predicts `churn` mainly from
`tenure`; `x2` and `score` each contribute less than 0.001 of model gain.
Holdout accuracy is 0.95 against a 0.77 majority baseline. A decision
table on `tenure` alone reaches cross-validated accuracy 0.896 against
0.769. These results support evaluating `tenure` as a predictor.

In [3]:
target = [f for f in report.findings
          if f["category"] == "dependencies" and f.get("type") == "predictive"
          and f["subject"].get("target") == "churn"]
for f in target:
    e = f["evidence"][0]
    m = e["metrics"]
    value = m.get("metric", m.get("cv_accuracy"))
    print(f["claim"])
    print("   inputs:", f["subject"]["inputs"],
          "| score:", round(value, 3),
          "| baseline:", round(m["baseline"], 3))

churn is predictable from tenure (100% of gain) (holdout accuracy 0.95 vs 0.77 majority baseline; 0.95 using all inputs); the share of rows with churn = 'true' is highest where tenure in (1, 3] (97%) and lowest where tenure in (24, 26] (2%)
   inputs: ['score', 'tenure'] | score: 0.95 | baseline: 0.774


## The control: shuffle the target

As a negative control, shuffle `churn` and profile the table again. The
predictive findings about `churn` should disappear. If they remain,
investigate possible leakage in the analysis.

In [4]:
permuted = df.with_columns(pl.col("churn").shuffle(seed=77))
permuted_report = onceover.profile(permuted)
kept = [f for f in permuted_report.findings
        if f["category"] == "dependencies" and f.get("type") == "predictive"
        and f["subject"].get("target") == "churn"]
print("predictive churn findings after shuffling:", len(kept))

predictive churn findings after shuffling: 0


No predictive finding about the shuffled target remains. This control
checks that the screen does not report predictability after the planted
target relationship has been removed.

## Decisions supported by the fixture


In [5]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Statement': ['`tenure` and `tenure_years` are exact functions of each other (0 violations)', '`score` predictable from `x2` and `x1` (holdout R² 0.98; 8 regimes, 0.910 vs 0.002 linear)', '`channel` 98.1% one value', '`segment → plan` 96–98%; `segment, score → plan` 99.22%', '`churn` from `tenure` (0.896 CV; holdout 0.95 vs 0.77)'], 'Evidence class': ['measured · full', 'estimated · holdout', 'measured · full', 'measured · diagnostic partition', 'estimated · holdout'], 'Trimming decision': ['drop the rounded copy', 'keep `x1`, `x2`; consider an explicit product', 'exclude from one-hot', 'inspect mapping violations and support', 'evaluate `tenure` as a predictor']})
(GT(_doc_table)
 .fmt_markdown(columns=['Statement', 'Evidence class', 'Trimming decision'])
 .tab_header(title='Decisions supported by the fixture')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


GT(_tbl_data=┌────────────────────────────────────┬────────────────────────┬────────────────────────────────────┐
│ Statement                          ┆ Evidence class         ┆ Trimming decision                  │
╞════════════════════════════════════╪════════════════════════╪════════════════════════════════════╡
│ `tenure` and `tenure_years` ar…    ┆ measured · full        ┆ drop the rounded copy              │
│ `score` predictable from `x2` …    ┆ estimated · holdout    ┆ keep `x1`, `x2`; consider an e…    │
│ `channel` 98.1% one value          ┆ measured · full        ┆ exclude from one-hot               │
│ `segment → plan` 96–98%; `segm…    ┆ measured · diagnostic  ┆ inspect mapping violations and…    │
│                                    ┆ partitio…              ┆                                    │
│ `churn` from `tenure` (0.896 C…    ┆ estimated · holdout    ┆ evaluate `tenure` as a predict…    │
└────────────────────────────────────┴────────────────────────┴────────────────────────────────────┘, _body=<great_tables._gt_data.Body object at 0x113d56510>, _boxhead=Boxhead([ColInfo(var='Statement', type=<ColInfoTypeEnum.default: 1>, column_label='Statement', column_align='left', column_width=None, stub_level=None), ColInfo(var='Evidence class', type=<ColInfoTypeEnum.default: 1>, column_label='Evidence class', column_align='left', column_width=None, stub_level=None), ColInfo(var='Trimming decision', type=<ColInfoTypeEnum.default: 1>, column_label='Trimming decision', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x113d56270>, _spanners=Spanners([]), _heading=Heading(title='Decisions supported by the fixture', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x113d56900>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x113d6e710>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Statement', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Statement', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Statement', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Statement', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Statement', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Evidence class', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Evidence class', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Evidence class', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Evidence class', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colna

The groups view links findings that share columns:


In [6]:
for g in report.groups:
    print(g["id"], "—", len(g["columns"]), "columns,", len(g["finding_ids"]), "findings")
    print("   columns:", ", ".join(g["columns"]))
    print("   ", g["description"])

group-1 — 9 columns, 19 findings
   columns: channel, churn, plan, score, segment, tenure, tenure_years, x1, x2
    Connected findings; inspect individual statements and directions before inferring redundancy.


The whole table is one connected component: the exact copy joins `tenure`
to `tenure_years`, the predictive statements join `score` to `x1` and
`x2`, and other shared-column links connect `churn` and `channel`.
One derived column can connect otherwise separate sets of findings
into a single component. Use groups to navigate shared columns; assess
redundancy from the individual findings.